# SI Figure 7

Compare the six growth and transfer-efficiency settings.

Input: generated/simulations.csv and generated/run_config.json, plus generated/sensitivity.csv. Run 00_generate_simulations.ipynb first. Set DATA_DIR for a different run.

Fit slopes separately for each setting.

## Settings

Change DATA_DIR and SAVE in the next cell.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

DATA_DIR = Path('generated')
SAVE = True
FIGURE_DIR = Path('figures')
FOCUS_DEGREE = 1.0
FOCUS_STRENGTH = 0.5
simulations = pd.read_csv(DATA_DIR / 'simulations.csv')
config = json.loads((DATA_DIR / 'run_config.json').read_text(encoding='utf-8'))
FUNCTION_COL = 'biomass_for_bef'
if config['profile'] != 'final_10000':
    raise ValueError('Use data from the final_10000 simulation notebook')

def slope_table(df, x='N', y=None, groups=('degree','strength')):
    y = FUNCTION_COL if y is None else y
    rows = []
    for keys, part in df.groupby(list(groups)):
        if part[x].nunique() < 2:
            continue
        fit = linregress(part[x], part[y])
        rows.append(dict(zip(groups, keys if isinstance(keys, tuple) else (keys,))) |
                    dict(slope=fit.slope, intercept=fit.intercept, stderr=fit.stderr,
                         n=len(part), richness_levels=part[x].nunique()))
    return pd.DataFrame(rows)

def set_strength_axis(ax, values):
    lo, hi = float(np.min(values)), float(np.max(values))
    ticks = sorted(set([lo, hi] + [float(x) for x in np.linspace(.2, 1, 5) if lo < x < hi]))
    ax.set_xlim(lo, hi)
    ax.set_xticks(ticks, [f'{x:g}' for x in ticks])

def finish(fig, stem):
    fig.tight_layout()
    if SAVE:
        FIGURE_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURE_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')
        fig.savefig(FIGURE_DIR / f'{stem}.pdf', bbox_inches='tight')
    plt.show()


## Figure

In [ ]:
sensitivity = pd.read_csv(DATA_DIR / 'sensitivity.csv')
coverage = sensitivity.groupby(['efficiency','growth_h']).size()
print('Sensitivity table:', sensitivity.shape, '\nRuns per setting:', coverage.to_dict())
settings = sorted(sensitivity[['efficiency','growth_h']].drop_duplicates().itertuples(index=False, name=None))
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for ax, (efficiency, growth_h) in zip(axes.flat, settings):
    subset = sensitivity[np.isclose(sensitivity.efficiency, efficiency) & np.isclose(sensitivity.growth_h, growth_h)]
    slopes = slope_table(subset)
    space = slopes.pivot(index='degree', columns='strength', values='slope').sort_index().sort_index(axis=1)
    image = ax.pcolormesh(space.columns.to_numpy(), space.index.to_numpy(), space.to_numpy(),
                          shading='nearest', cmap='magma_r', vmin=0)
    ax.set_title(f'efficiency={efficiency:g}, growth H={growth_h:.3g}')
    ax.set(xlabel='Strength', ylabel='Degree')
    set_strength_axis(ax, space.columns.to_numpy())
fig.colorbar(image, ax=axes.ravel().tolist(), label='BEF slope', shrink=.7)
if SAVE:
    FIGURE_DIR.mkdir(exist_ok=True)
    fig.savefig(FIGURE_DIR / 'si_figure_7.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURE_DIR / 'si_figure_7.pdf', bbox_inches='tight')
plt.show()